# Notebook 06 — Autoencoder (Detekcja Anomalii)

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

**Nienadzorowana detekcja anomalii** przy użyciu Autoencodera.

### Idea Autoencodera
Autoencoder to sieć neuronowa, która:
1. **Enkoduje** dane wejściowe do niskowymiarowej reprezentacji (bottleneck)
2. **Dekoduje** reprezentację z powrotem do przestrzeni oryginalnej
3. Minimalizuje **błąd rekonstrukcji** (MSE między wejściem a wyjściem)

Kluczowa właściwość: model trenowany tylko na **normalnych** dniach (target=0) nauczy się rekonstruować normalne wzorce. Dla anomalii (kryzysy, szoki) błąd rekonstrukcji będzie wysoki → sygnał anomalii.

### Wynik: nowe cechy
- `autoencoder_re` — błąd rekonstrukcji (ciągły)
- `autoencoder_anomaly` — 1 jeśli RE > kwantyl 95% (binarny)

In [1]:
# ============================================================
# Konfiguracja wspolna
# ============================================================
import os, warnings
os.environ.setdefault('KERAS_BACKEND', 'torch')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
warnings.filterwarnings('ignore')
import importlib
import pandas as pd, numpy as np
import pipeline_lib as pl, models_lib as ml, run_analysis as ra
for m in (pl, ml, ra): importlib.reload(m)
pd.set_option('display.width', 160)
print(f'Ziarno losowe: {pl.SEED} | horyzont: {pl.HORIZON} sesji | embargo: {pl.EMBARGO} sesji')


Ziarno losowe: 42 | horyzont: 5 sesji | embargo: 10 sesji


In [2]:
# ============================================================
# Autoenkoder: nienadzorowana detekcja anomalii (analiza opisowa)
# ============================================================
yearly = ra.autoencoder_report()
print(yearly.to_string())
print('\nKonfiguracja:')
print(pd.read_csv('results/tables/06_autoencoder_config.csv').to_string(index=False))


      Sesje  Anomalie AE  Zdarzenia stresowe  Pokrycie  Precyzja AE  Czułość AE
Rok                                                                            
2019     47            1                   0         0        0.000       0.000
2020    252           49                  49        36        0.735       0.735
2021    251           10                   9         5        0.500       0.556
2022    251           33                  58        17        0.515       0.293
2023    250           12                   7         0        0.000       0.000
2024    249           62                   0         0        0.000       0.000
2025    249           31                   6         6        0.194       1.000
2026    156           18                   2         0        0.000       0.000

Konfiguracja:
 Próg anomalii (95. percentyl)  Liczba anomalii  Udział anomalii (%)  Wymiar warstwy wąskiej  Liczba cech wejściowych
                       0.61031              216                12.6